# Object Image Classifier: Transfer Learning vs. Training From Scratch — Final Experiments
**COMP 365 Individual Project · Final Submission** · Ryan Kevena

**Question.** On low-resolution (32×32) CIFAR-10 images, how much does fine-tuning an ImageNet-pretrained ResNet-18 improve test accuracy over the same ResNet-18 trained from scratch, how does that gap change with training-set size, and does fine-tuning beat a frozen-feature linear probe?

**What changed since Checkpoint 2 (from instructor feedback)**
1. **Validation split.** 5,000 training images (500 per class) are held out as a validation set. All learning rates and regularization values are chosen on it. The test set is used only for the final numbers.
2. **Two clearly separated experiments.**
   * **Experiment A – fixed 200 steps** for every training size, with the Checkpoint 2 learning rates (re-run on the official data with 3 seeds).
   * **Experiment B – equal epochs** (`EPOCHS` passes over the training subset for every size), with learning rates chosen on the validation split.
3. **Safe caching.** Each saved run stores the dataset source, a fingerprint of the images and their order, the exact training-image IDs, the preprocessing, and every training setting. The file name contains a hash of all of this, so changing anything forces a fresh run.
4. **All four methods kept** in the final comparison (fine-tuned, scratch, linear probe, pixel logistic regression). Learning rates for both neural methods are saved in `outputs/final_settings.csv`.
5. **Final app weights.** The model with the best *validation* accuracy is exported to `outputs/final_model.pt` and used by the web app and the phone-photo test.
6. **Pinned versions.** The notebook writes the exact package versions it ran with to `requirements.txt` and `outputs/environment.json`.

**How to run (Google Colab):** `Runtime → Change runtime type → T4 GPU`, then `Runtime → Run all`. Set `QUICK_TEST = True` first if you want a fast check that everything works.

In [ ]:
# ---- Install the two packages Colab does not include (exact versions are recorded at the end) ----
import importlib.util, subprocess, sys, os
for pkg in ["gradio", "gradio_client"]:
    if importlib.util.find_spec(pkg) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
missing = [f for f in ["app.py", "classifier.py"] if not os.path.exists(f)]
if missing:
    print("WARNING: these files are missing from the working folder:", missing,
          "- upload them (or clone the repository) before running Section 10, or set TEST_WEB_APP = False.")

In [ ]:
# ---- Experiment settings (every value here is saved with each run) ----
QUICK_TEST = False              # True = tiny version to check the pipeline (a few minutes). Use False for the paper.

SIZES = [500, 1000, 2500, 5000] # balanced training-set sizes (50 / 100 / 250 / 500 per class)
SEEDS = [0, 1]                  # 2 seeds: different training subsets and initializations (fast version)
VAL_PER_CLASS = 500             # validation images per class, taken from the 50,000 training images
VAL_SEED = 12345                # fixed, so the validation split never changes
BATCH_SIZE = 64
IMG_SIZE = 64                   # 32x32 images upsampled 2x so the ImageNet filters apply better
WEIGHT_DECAY = 5e-4
LABEL_SMOOTHING = 0.1

# Experiment A: fixed number of optimizer steps for every size (Checkpoint 2 recipe)
FIXED_STEPS = 200
LR_FIXED = {"scratch": 0.05, "finetune": 0.01}

# Experiment B: equal epochs for every size; learning rates chosen on the validation split
EPOCHS = 10
LR_GRID = {"scratch": [0.05, 0.1], "finetune": [0.003, 0.01]}
BACKBONE_LR_MULT_GRID = [0.1, 1.0]   # fine-tuning only: pretrained layers use lr x this value
C_GRID = [0.01, 0.1, 1.0]            # logistic-regression regularization for both linear baselines

REQUIRE_OFFICIAL_DATA = True    # stop with an error instead of silently using the JPEG mirror
TEST_WEB_APP = True             # launch app.py and query it through its API (section 9)
OUT_DIR = "outputs"
CODE_VERSION = "final-v1"       # change this if you edit the training code, to invalidate old runs

if QUICK_TEST:
    SIZES, SEEDS, EPOCHS, FIXED_STEPS = [500, 1000], [0], 2, 20
    LR_GRID = {"scratch": [0.05], "finetune": [0.01]}; BACKBONE_LR_MULT_GRID = [1.0]; C_GRID = [0.1]
    OUT_DIR = "outputs_quick"
N_MAX = max(SIZES)

In [ ]:
import os, time, random, glob, io, zipfile, urllib.request, warnings, json, hashlib, math, platform
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.nn.functional as F
import torchvision
from torchvision.models import resnet18
from sklearn.metrics import f1_score, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from scipy.stats import chi2
from PIL import Image

os.makedirs(OUT_DIR, exist_ok=True)
RUN_DIR = f"{OUT_DIR}/runs"; os.makedirs(RUN_DIR, exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_amp = device.type == "cuda"
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
print("PyTorch", torch.__version__, "| torchvision", torchvision.__version__, "| device:", device,
      "|", torch.cuda.get_device_name(0) if device.type == "cuda" else platform.processor())
if device.type != "cuda" and not QUICK_TEST:
    print("WARNING: no GPU found. The full run is very slow on a CPU. In Colab choose Runtime -> Change runtime type -> T4 GPU.")

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

def short_hash(*objs):
    """Stable 12-character hash of numpy arrays (contents AND order) or JSON-serializable settings."""
    h = hashlib.sha256()
    for obj in objs:
        if isinstance(obj, np.ndarray):
            a = np.ascontiguousarray(obj); h.update(str((a.shape, a.dtype.str)).encode()); h.update(a.tobytes())
        else:
            h.update(json.dumps(obj, sort_keys=True).encode())
    return h.hexdigest()[:12]

## 1. Data: CIFAR-10
The notebook loads CIFAR-10 from the Hugging Face copy of the official files (fast in Colab), or from the official `torchvision` download if that fails. A fingerprint of all images, labels and their order (`DATA_ID`) is computed and stored with every run, so results from a different copy or order of the data can never be reused by mistake.

In [ ]:
CLASSES = ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']

def load_official():
    tr = torchvision.datasets.CIFAR10(root="data", train=True, download=True)
    te = torchvision.datasets.CIFAR10(root="data", train=False, download=True)
    return tr.data, np.array(tr.targets), te.data, np.array(te.targets)

def load_mirror():
    root = "CIFAR-10-images-master"
    if not os.path.isdir(root):
        url = "https://codeload.github.com/YoongiKim/CIFAR-10-images/zip/refs/heads/master"
        zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(url).read())).extractall(".")
    out = []
    for split in ["train", "test"]:
        X, y = [], []
        for c, name in enumerate(CLASSES):
            for f in sorted(glob.glob(f"{root}/{split}/{name}/*.jpg")):
                X.append(np.array(Image.open(f).convert("RGB"))); y.append(c)
        out += [np.stack(X), np.array(y)]
    return out

def load_hf():
    # Fast in Colab: lossless (PNG) copy of the official CIFAR-10 files on Hugging Face
    from datasets import load_dataset
    ds = load_dataset("uoft-cs/cifar10")
    out = []
    for split in ["train", "test"]:
        out += [np.stack([np.array(im.convert("RGB")) for im in ds[split]["img"]]), np.array(ds[split]["label"])]
    return out

try:
    Xtr_np, ytr_np, Xte_np, yte_np = load_hf(); DATA_SOURCE = "CIFAR-10, Hugging Face uoft-cs/cifar10 (lossless copy of official files)"
except Exception as e0:
    print("Hugging Face copy unavailable:", type(e0).__name__, "-> trying the official torchvision download")
    Xtr_np = None
try:
    if Xtr_np is None:
        Xtr_np, ytr_np, Xte_np, yte_np = load_official(); DATA_SOURCE = "official CIFAR-10 (torchvision)"
except Exception as e:
    if REQUIRE_OFFICIAL_DATA:
        raise RuntimeError("Official CIFAR-10 download failed. Try again, or set REQUIRE_OFFICIAL_DATA = False "
                           "to use the JPEG mirror (results will be stored separately).") from e
    print("Official download unavailable -> using GitHub JPEG mirror")
    Xtr_np, ytr_np, Xte_np, yte_np = load_mirror(); DATA_SOURCE = "CIFAR-10 JPEG mirror (GitHub)"

ytr_np = ytr_np.astype(np.int64); yte_np = yte_np.astype(np.int64)
DATA_ID = short_hash(Xtr_np, ytr_np, Xte_np, yte_np)   # changes if any image, label or the image order changes
Xtr = torch.tensor(Xtr_np).permute(0, 3, 1, 2).contiguous(); ytr = torch.tensor(ytr_np)
Xte = torch.tensor(Xte_np).permute(0, 3, 1, 2).contiguous(); yte = torch.tensor(yte_np)
print("Data source:", DATA_SOURCE, "| fingerprint:", DATA_ID)
print("Train:", tuple(Xtr.shape), "| Test:", tuple(Xte.shape))
assert Xtr.shape[1:] == (3, 32, 32) and Xte.shape[1:] == (3, 32, 32)
assert np.bincount(ytr_np).min() == 5000 and np.bincount(yte_np).min() == 1000

## 2. Validation split, balanced training subsets and preprocessing
* **Validation:** 500 images per class (5,000 total) are removed from the training set with a fixed seed. They are never used for training.
* **Training subsets:** drawn only from the remaining 45,000 images. For each seed, smaller subsets are nested inside larger ones.
* **Test set:** the official 10,000 test images, used only for the final reported numbers.

In [ ]:
rng_val = np.random.default_rng(VAL_SEED)
val_idx = np.concatenate([rng_val.permutation(np.where(ytr_np == c)[0])[:VAL_PER_CLASS] for c in range(10)])
pool_idx = np.setdiff1d(np.arange(len(ytr_np)), val_idx)
VAL_ID = short_hash(val_idx)

def balanced_subset(n_total, seed):
    """Balanced subset from the training pool. Same seed -> nested subsets for different sizes."""
    rng = np.random.default_rng(seed)
    return np.concatenate([rng.permutation(pool_idx[ytr_np[pool_idx] == c])[:n_total // 10] for c in range(10)])

for s in SEEDS:   # checks: balanced, nested, and no overlap with validation
    big = balanced_subset(N_MAX, s)
    assert np.bincount(ytr_np[big], minlength=10).min() == N_MAX // 10
    assert len(np.intersect1d(big, val_idx)) == 0
    assert set(balanced_subset(min(SIZES), s)) <= set(big)
print(f"Validation: {len(val_idx)} images | training pool: {len(pool_idx)} images | checks passed")

MEAN = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
STD  = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)
PREPROC = dict(img_size=IMG_SIZE, train="reflect-pad 4, random 32x32 crop, horizontal flip, bilinear upsample",
               eval="bilinear upsample", normalize="ImageNet mean/std")

def gpu_augment(x_uint8):
    x = x_uint8.float() / 255.0; B = x.size(0)
    x = F.pad(x, (4, 4, 4, 4), mode="reflect")
    i = torch.randint(0, 9, (B,), device=x.device); j = torch.randint(0, 9, (B,), device=x.device)
    ar = torch.arange(32, device=x.device)
    rows = (i[:, None] + ar)[:, None, :, None]; cols = (j[:, None] + ar)[:, None, None, :]
    bi = torch.arange(B, device=x.device)[:, None, None, None]; ci = torch.arange(3, device=x.device)[None, :, None, None]
    x = x[bi, ci, rows, cols]
    flip = torch.rand(B, device=x.device) < 0.5
    x = torch.where(flip[:, None, None, None], x.flip(3), x)
    x = F.interpolate(x, size=IMG_SIZE, mode="bilinear", align_corners=False)
    return (x - MEAN) / STD

def prep_eval(x_uint8):
    x = F.interpolate(x_uint8.float() / 255.0, size=IMG_SIZE, mode="bilinear", align_corners=False)
    return (x - MEAN) / STD

## 3. Models, training and safe caching
* **Scratch:** ResNet-18 (torchvision architecture) with random initialization.
* **Fine-tuned:** the same ResNet-18 starting from ImageNet-1k weights (`resnet18_a1_0` from timm). The new 10-class layer uses learning rate `lr`; the pretrained layers use `lr × backbone_lr_mult`.
* Optimizer: SGD (momentum 0.9, Nesterov), one-cycle learning-rate schedule, label smoothing 0.1, mixed precision on GPU. Batches are drawn by shuffling the subset (each image once per pass).

In [ ]:
PRETRAINED_NAME = "timm resnet18_a1_0-d63eafa0.pth"
PRETRAINED_URL = ("https://github.com/huggingface/pytorch-image-models/releases/download/"
                  "v0.1-rsb-weights/resnet18_a1_0-d63eafa0.pth")
IMAGENET_STATE = torch.hub.load_state_dict_from_url(PRETRAINED_URL, map_location="cpu", check_hash=True)

def make_model(kind):
    m = resnet18(weights=None)
    if kind == "finetune":
        m.load_state_dict(IMAGENET_STATE)
    m.fc = nn.Linear(m.fc.in_features, 10)
    return m.to(device).to(memory_format=torch.channels_last)

@torch.no_grad()
def predict_probs(model, X, bs=500):
    model.eval(); probs = []
    for k in range(0, len(X), bs):
        xb = prep_eval(X[k:k + bs].to(device)).contiguous(memory_format=torch.channels_last)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
            probs.append(F.softmax(model(xb).float(), 1).cpu())
    return torch.cat(probs)

def steps_equal_epochs(n):
    return EPOCHS * math.ceil(n / BATCH_SIZE)

def train_nn(kind, idx, seed, lr, backbone_mult, total_steps):
    set_seed(seed)
    Xs, ys = Xtr[idx].to(device), ytr[idx].to(device); n = len(idx)
    model = make_model(kind)
    head = list(model.fc.parameters()); head_ids = {id(p) for p in head}
    backbone = [p for p in model.parameters() if id(p) not in head_ids]
    opt = torch.optim.SGD([{"params": backbone, "lr": lr * backbone_mult}, {"params": head, "lr": lr}],
                          lr=lr, momentum=0.9, nesterov=True, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=[lr * backbone_mult, lr], total_steps=total_steps, pct_start=0.15)
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    g = torch.Generator().manual_seed(seed)
    def batches():
        while True:
            perm = torch.randperm(n, generator=g)
            for k in range(0, n, BATCH_SIZE):
                yield perm[k:k + BATCH_SIZE]
    it = batches(); model.train(); t0 = time.time()
    for step in range(total_steps):
        b = next(it).to(device)
        xb = gpu_augment(Xs[b]).contiguous(memory_format=torch.channels_last)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
            loss = F.cross_entropy(model(xb), ys[b], label_smoothing=LABEL_SMOOTHING)
        opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
    return model, time.time() - t0, float(loss.item())

def run_nn(exp, kind, n, seed, lr, backbone_mult, steps, eval_test=True, keep_weights=False):
    """Train one network, or reuse a saved run ONLY if every setting matches exactly."""
    idx = balanced_subset(n, seed)
    cfg = dict(code=CODE_VERSION, exp=exp, method=kind, n_train=n, seed=seed, data_source=DATA_SOURCE, data_id=DATA_ID,
               val_id=VAL_ID, train_ids=short_hash(idx), preproc=PREPROC, lr=lr, backbone_lr_mult=backbone_mult,
               steps=steps, batch_size=BATCH_SIZE, weight_decay=WEIGHT_DECAY, label_smoothing=LABEL_SMOOTHING,
               pretrained=PRETRAINED_NAME if kind == "finetune" else None, eval_test=eval_test)
    key = short_hash(cfg)
    path = f"{RUN_DIR}/{exp}_{kind}_n{n}_seed{seed}_{key}.pt"; wpath = path.replace(".pt", "_weights.pt")
    if os.path.exists(path) and (not keep_weights or os.path.exists(wpath)):
        rec = torch.load(path, weights_only=False)
        if rec["config"] == cfg:
            return rec, True
    model, sec, last_loss = train_nn(kind, idx, seed, lr, backbone_mult, steps)
    val_pred = predict_probs(model, Xtr[val_idx]).argmax(1).numpy()
    rec = dict(config=cfg, exp=exp, method=kind, n_train=n, seed=seed, lr=lr, backbone_lr_mult=backbone_mult,
               backbone_lr=lr * backbone_mult, C=None,
               steps=steps, epochs_seen=steps * BATCH_SIZE / n, val_acc=float((val_pred == ytr_np[val_idx]).mean()),
               final_train_loss=last_loss, train_sec=sec)
    if eval_test:
        probs = predict_probs(model, Xte); pred = probs.argmax(1).numpy()
        rec.update(test_acc=float((pred == yte_np).mean()), macro_f1=float(f1_score(yte_np, pred, average="macro")),
                   probs=probs.half())
    if keep_weights:
        torch.save({k: v.cpu() for k, v in model.state_dict().items()}, wpath); rec["weights_path"] = wpath
    torch.save(rec, path)
    return rec, False

def show(rec, cached):
    t = f"test {rec['test_acc']*100:5.1f}%" if "test_acc" in rec else "test (not evaluated)"
    print(f"{rec['exp']} | seed {rec['seed']} | {rec['method']:>8} | n={rec['n_train']:>5} | lr {rec['lr']:<6} "
          f"backbone lr {rec['backbone_lr']:<7.4g} | steps {rec['steps']:>5} | val {rec['val_acc']*100:5.1f}% | {t}"
          f"{' (saved run)' if cached else ''}", flush=True)

## 4. Linear baselines (same subsets, same validation and test images)
* **Pixel logistic regression:** logistic regression on the 3,072 raw pixel values.
* **Linear probe:** the ImageNet ResNet-18 is *frozen*; logistic regression is trained on its 512-number feature vector.

In [ ]:
@torch.no_grad()
def frozen_features(X, bs=500):
    m = resnet18(weights=None); m.load_state_dict(IMAGENET_STATE); m.fc = nn.Identity()
    m = m.to(device).eval(); out = []
    for k in range(0, len(X), bs):
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
            out.append(m(prep_eval(X[k:k + bs].to(device))).float().cpu())
    return torch.cat(out).numpy()

t0 = time.time()
feat_tr = frozen_features(Xtr); feat_te = frozen_features(Xte)
pix_tr = Xtr_np.reshape(len(Xtr_np), -1).astype(np.float32) / 255.0
pix_te = Xte_np.reshape(len(Xte_np), -1).astype(np.float32) / 255.0
print(f"Frozen ImageNet features for {len(feat_tr)} train + {len(feat_te)} test images in {time.time()-t0:.0f}s")

def run_linear(kind, n, seed, C, eval_test=True):
    idx = balanced_subset(n, seed)
    Xa, Xt = (feat_tr, feat_te) if kind == "linear_probe" else (pix_tr, pix_te)
    t1 = time.time()
    sc = StandardScaler().fit(Xa[idx])
    clf = LogisticRegression(C=C, max_iter=1000).fit(sc.transform(Xa[idx]), ytr_np[idx])
    rec = dict(exp=None, method=kind, n_train=n, seed=seed, lr=None, backbone_lr_mult=None, backbone_lr=None, C=C, steps=None, epochs_seen=None,
               val_acc=float((clf.predict(sc.transform(Xa[val_idx])) == ytr_np[val_idx]).mean()), train_sec=time.time() - t1)
    if eval_test:
        probs = clf.predict_proba(sc.transform(Xt)); pred = probs.argmax(1)
        rec.update(test_acc=float((pred == yte_np).mean()), macro_f1=float(f1_score(yte_np, pred, average="macro")),
                   probs=torch.tensor(probs).half())
    return rec, (sc, clf)

## 5. Choosing settings on the validation split (test set not used)
Every candidate is trained on the seed-0 subset of the largest size with the Experiment B budget (equal epochs) and scored **only on validation**. The best value for each method is then used for all sizes and seeds in Experiment B.

In [ ]:
tune_rows = []
for lr in LR_GRID["scratch"]:
    rec, c = run_nn("tune", "scratch", N_MAX, 0, lr, 1.0, steps_equal_epochs(N_MAX), eval_test=False); show(rec, c); tune_rows.append(rec)
for lr in LR_GRID["finetune"]:
    for mult in BACKBONE_LR_MULT_GRID:
        rec, c = run_nn("tune", "finetune", N_MAX, 0, lr, mult, steps_equal_epochs(N_MAX), eval_test=False); show(rec, c); tune_rows.append(rec)
for kind in ["linear_probe", "pixel_logreg"]:
    for C in C_GRID:
        rec, _ = run_linear(kind, N_MAX, 0, C, eval_test=False); tune_rows.append(rec)
        print(f"tune | {kind:>12} | C={C:<5} | val {rec['val_acc']*100:5.1f}%", flush=True)

tune_df = pd.DataFrame([{k: r[k] for k in ["method", "lr", "backbone_lr_mult", "backbone_lr", "C", "steps", "val_acc"]} for r in tune_rows])
tune_df.to_csv(f"{OUT_DIR}/final_validation_search.csv", index=False)
best = {m: tune_df[tune_df.method == m].sort_values("val_acc", ascending=False).iloc[0] for m in tune_df.method.unique()}
BEST_NN = {m: dict(lr=float(best[m].lr), mult=float(best[m].backbone_lr_mult)) for m in ["scratch", "finetune"]}
BEST_C = {m: float(best[m].C) for m in ["linear_probe", "pixel_logreg"]}
print("\nChosen on validation:", BEST_NN, BEST_C)
display(tune_df)

## 6. Main experiments: 2 budgets × 2 networks × 4 sizes × 3 seeds, plus both linear baselines
* **Experiment A (fixed steps):** 200 steps for every size; Checkpoint 2 learning rates (scratch 0.05, fine-tuned 0.01, same rate for all layers).
* **Experiment B (equal epochs):** `EPOCHS` passes over the subset for every size; learning rates chosen in Section 5.
* The linear baselines do not use steps or epochs, so the same baseline numbers appear in both comparisons.

In [ ]:
results, all_probs, linear_models = [], {}, {}
for seed in SEEDS:
    for n in SIZES:
        for kind in ["scratch", "finetune"]:
            rec, c = run_nn("A_fixed_steps", kind, n, seed, LR_FIXED[kind], 1.0, FIXED_STEPS); show(rec, c)
            all_probs[("A_fixed_steps", kind, n, seed)] = rec.pop("probs"); results.append(rec)
            keep = (n == N_MAX and seed == 0)
            rec, c = run_nn("B_equal_epochs", kind, n, seed, BEST_NN[kind]["lr"], BEST_NN[kind]["mult"],
                            steps_equal_epochs(n), keep_weights=keep); show(rec, c)
            all_probs[("B_equal_epochs", kind, n, seed)] = rec.pop("probs"); results.append(rec)
        for kind in ["linear_probe", "pixel_logreg"]:
            rec, model = run_linear(kind, n, seed, BEST_C[kind])
            if n == N_MAX and seed == 0: linear_models[kind] = model
            probs = rec.pop("probs")
            for exp in ["A_fixed_steps", "B_equal_epochs"]:
                all_probs[(exp, kind, n, seed)] = probs; results.append({**rec, "exp": exp})
            print(f"   seed {seed} | {kind:>12} | n={n:>5} | C={BEST_C[kind]} | val {rec['val_acc']*100:5.1f}% | test {rec['test_acc']*100:5.1f}%")

COLS = ["exp", "method", "n_train", "seed", "lr", "backbone_lr", "C", "steps", "epochs_seen", "val_acc", "test_acc", "macro_f1", "train_sec"]
df = pd.DataFrame([{k: r.get(k) for k in COLS} for r in results])
df.to_csv(f"{OUT_DIR}/final_all_runs.csv", index=False)

## 7. Results: settings, mean ± std over seeds, confidence intervals and paired tests

In [ ]:
ORDER = ["finetune", "scratch", "linear_probe", "pixel_logreg"]
NAMES = {"finetune": "Fine-tuned ResNet-18", "scratch": "ResNet-18 from scratch",
         "linear_probe": "Linear probe (frozen ImageNet)", "pixel_logreg": "Pixel logistic regression"}
EXPS = {"A_fixed_steps": f"Experiment A: fixed {FIXED_STEPS} steps", "B_equal_epochs": f"Experiment B: equal epochs ({EPOCHS})"}

# Settings table: both learning rates for both neural methods, in both experiments
settings = (df[df.method.isin(["finetune", "scratch"])].groupby(["exp", "method", "n_train"])
            .agg(head_lr=("lr", "first"), backbone_lr=("backbone_lr", "first"), steps=("steps", "first"),
                 epochs_seen=("epochs_seen", "first")).reset_index())
settings["epochs_seen"] = settings["epochs_seen"].round(2)
lin = pd.DataFrame([{"method": m, "C": BEST_C[m]} for m in BEST_C])
settings.to_csv(f"{OUT_DIR}/final_settings.csv", index=False); lin.to_csv(f"{OUT_DIR}/final_settings_linear.csv", index=False)
print("Training settings (head = new 10-class layer; backbone = all other layers):"); display(settings); display(lin)

summaries = {}
for exp, title in EXPS.items():
    d = df[df.exp == exp]
    agg = d.groupby(["method", "n_train"])["test_acc"].agg(["mean", "std"]) * 100
    tab = pd.DataFrame({NAMES[m]: [f"{agg.loc[(m, n), 'mean']:.1f} ± {agg.loc[(m, n), 'std']:.1f}" for n in SIZES] for m in ORDER},
                       index=pd.Index(SIZES, name="training images"))
    for a, b, name in [("finetune", "scratch", "Gap: fine-tuned − scratch"), ("finetune", "linear_probe", "Gap: fine-tuned − linear probe")]:
        g = {n: (d[(d.method == a) & (d.n_train == n)].set_index("seed").test_acc
                 - d[(d.method == b) & (d.n_train == n)].set_index("seed").test_acc) * 100 for n in SIZES}
        tab[name] = [f"{g[n].mean():+.1f} ± {g[n].std():.1f}" for n in SIZES]
    summaries[exp] = agg
    print(f"\n{title}. Test accuracy (%), mean ± std over {len(SEEDS)} seeds, 10,000 test images. Chance = 10%.")
    display(tab); tab.to_csv(f"{OUT_DIR}/final_table_{exp}.csv")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
styles = {"finetune": ("o", "-"), "scratch": ("s", "-"), "linear_probe": ("^", "--"), "pixel_logreg": ("D", ":")}
for ax, (exp, title) in zip(axes, EXPS.items()):
    agg = summaries[exp]
    for m in ORDER:
        mu = [agg.loc[(m, n), "mean"] for n in SIZES]; sd = [agg.loc[(m, n), "std"] for n in SIZES]
        ax.errorbar(SIZES, mu, yerr=sd, marker=styles[m][0], ls=styles[m][1], lw=2, capsize=3, label=NAMES[m])
    ax.axhline(10, color="gray", ls="--", lw=1, label="Chance (10%)")
    ax.set_xscale("log"); ax.set_xticks(SIZES); ax.set_xticklabels([f"{n:,}" for n in SIZES]); ax.minorticks_off()
    ax.set_ylim(0, 80); ax.set_xlabel("Training images (balanced, 10 classes)"); ax.set_title(title); ax.grid(alpha=0.3)
axes[0].set_ylabel("Test accuracy (%)"); axes[1].legend(fontsize=8, loc="lower right")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/final_results_figure.png", dpi=200); plt.show()

In [ ]:
# Bootstrap 95% CIs and paired McNemar tests at the largest size, Experiment B
EXP = "B_equal_epochs"
rng = np.random.default_rng(0); N = len(yte_np); boots = rng.integers(0, N, (1000, N))
correct = {(m, s): all_probs[(EXP, m, N_MAX, s)].float().argmax(1).numpy() == yte_np for m in ORDER for s in SEEDS}
ci = pd.DataFrame([dict(method=NAMES[m], seed=s, acc=correct[(m, s)].mean() * 100,
                        ci_low=np.percentile(correct[(m, s)][boots].mean(1) * 100, 2.5),
                        ci_high=np.percentile(correct[(m, s)][boots].mean(1) * 100, 97.5)) for m in ORDER for s in SEEDS]).round(1)
ci.to_csv(f"{OUT_DIR}/final_bootstrap_ci_n{N_MAX}.csv", index=False); display(ci)

mc = []
for other in ["scratch", "linear_probe"]:
    for s in SEEDS:
        a, b = correct[("finetune", s)], correct[(other, s)]
        b01, b10 = int((a & ~b).sum()), int((~a & b).sum())
        stat = (abs(b01 - b10) - 1) ** 2 / max(b01 + b10, 1)
        mc.append(dict(comparison=f"fine-tuned vs {other}", seed=s, only_finetune_right=b01, only_other_right=b10,
                       mcnemar_chi2=round(stat, 1), p_value=chi2.sf(stat, 1)))
mc = pd.DataFrame(mc); mc.to_csv(f"{OUT_DIR}/final_mcnemar_n{N_MAX}.csv", index=False)
print(f"Paired McNemar tests on the same 10,000 test images ({EXP}, n={N_MAX}):"); display(mc)

## 8. Weak cases: per-class accuracy, confusion pairs and the most confident mistakes (Experiment B, largest size)

In [ ]:
pc = []
for m in ["finetune", "scratch", "linear_probe"]:
    for s in SEEDS:
        p = all_probs[(EXP, m, N_MAX, s)].float().argmax(1).numpy()
        for c in range(10):
            pc.append(dict(method=m, cls=CLASSES[c], seed=s, acc=(p[yte_np == c] == c).mean() * 100))
pc = pd.DataFrame(pc).groupby(["cls", "method"]).acc.mean().unstack("method").round(1).sort_values("finetune")
display(pc); pc.to_csv(f"{OUT_DIR}/final_per_class_n{N_MAX}.csv")

p0 = all_probs[(EXP, "finetune", N_MAX, 0)].float(); pred0 = p0.argmax(1).numpy(); conf0 = p0.max(1).values.numpy()
cm = confusion_matrix(yte_np, pred0)
pairs = sorted([(cm[i, j], CLASSES[i], CLASSES[j]) for i in range(10) for j in range(10) if i != j], reverse=True)[:6]
conf_pairs = pd.DataFrame(pairs, columns=["count (of 1,000)", "true", "predicted"])
display(conf_pairs); conf_pairs.to_csv(f"{OUT_DIR}/final_top_confusions.csv", index=False)

wrong = np.where(pred0 != yte_np)[0]; worst = wrong[np.argsort(-conf0[wrong])][:8]
fig, axes = plt.subplots(1, 8, figsize=(13, 2.3))
for ax, i in zip(axes, worst):
    ax.imshow(Xte_np[i]); ax.axis("off")
    ax.set_title(f"true: {CLASSES[yte_np[i]]}\npred: {CLASSES[pred0[i]]} ({conf0[i]*100:.0f}%)", fontsize=8, color="red")
plt.suptitle(f"Most confident mistakes: fine-tuned ResNet-18, Experiment B, n={N_MAX}, seed 0", fontsize=10)
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/final_confident_mistakes.png", dpi=170); plt.show()
print(f"Mean confidence when right: {conf0[pred0 == yte_np].mean()*100:.1f}% | when wrong: {conf0[wrong].mean()*100:.1f}%")

## 9. Final model for the web app (chosen by validation accuracy, not test accuracy)
Candidates: the three ResNet-18-based models from Experiment B at the largest size, seed 0. The linear probe is turned into an ordinary ResNet-18 by folding the feature scaling and logistic-regression weights into the last layer, so the app can load any of the three in the same way.

In [ ]:
dB = df[(df.exp == EXP) & (df.n_train == N_MAX) & (df.seed == 0)].set_index("method")
cands = ["finetune", "scratch", "linear_probe"]
FINAL_METHOD = max(cands, key=lambda m: dB.loc[m, "val_acc"])
print("Validation accuracy of candidates:", {m: round(dB.loc[m, "val_acc"] * 100, 1) for m in cands}, "-> final:", FINAL_METHOD)

if FINAL_METHOD == "linear_probe":
    sc, clf = linear_models["linear_probe"]
    W = clf.coef_ / sc.scale_[None, :]; b = clf.intercept_ - (clf.coef_ * (sc.mean_ / sc.scale_)[None, :]).sum(1)
    final_sd = {k: v for k, v in IMAGENET_STATE.items() if not k.startswith("fc.")}
    final_sd["fc.weight"] = torch.tensor(W, dtype=torch.float32); final_sd["fc.bias"] = torch.tensor(b, dtype=torch.float32)
else:
    w = [r for r in results if r["exp"] == EXP and r["method"] == FINAL_METHOD and r["n_train"] == N_MAX and r["seed"] == 0][0]
    final_sd = torch.load(w["weights_path"], map_location="cpu")

final_model = resnet18(weights=None); final_model.fc = nn.Linear(512, 10); final_model.load_state_dict(final_sd)
final_model = final_model.to(device).to(memory_format=torch.channels_last)
final_pred = predict_probs(final_model, Xte).argmax(1).numpy()
ref_pred = all_probs[(EXP, FINAL_METHOD, N_MAX, 0)].float().argmax(1).numpy()
agree = (final_pred == ref_pred).mean() * 100
print(f"Exported model test accuracy {(final_pred == yte_np).mean()*100:.1f}% | agrees with the experiment's predictions on {agree:.2f}% of test images")
assert agree > 99.0, "exported model does not match the experiment"
torch.save({k: v.cpu() for k, v in final_model.state_dict().items()}, f"{OUT_DIR}/final_model.pt")
json.dump(dict(method=FINAL_METHOD, experiment=EXP, n_train=N_MAX, seed=0, img_size=IMG_SIZE, classes=CLASSES,
               mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225], data_source=DATA_SOURCE, data_id=DATA_ID,
               val_acc=float(dB.loc[FINAL_METHOD, "val_acc"]), test_acc=float((final_pred == yte_np).mean())),
          open(f"{OUT_DIR}/final_model.json", "w"), indent=2)
print("Saved", f"{OUT_DIR}/final_model.pt", "and", f"{OUT_DIR}/final_model.json")

## 10. Web-app test: launch `app.py` with the final model and query it through its API
50 test images (5 per class) are saved as PNG files and sent to the running app, exactly like a user upload. The real phone-photo test is separate: run `python test_phone_photos.py` after adding photos to `phone_photos/` (see README). Those results are **not** CIFAR-10 results and must be reported separately.

In [ ]:
import subprocess, sys
if TEST_WEB_APP:
    os.makedirs("app_test", exist_ok=True)
    sel = np.concatenate([np.where(yte_np == c)[0][:5] for c in range(10)])
    for i in sel: Image.fromarray(Xte_np[i]).save(f"app_test/{i}.png")
    server = subprocess.Popen([sys.executable, "app.py", "--weights", f"{OUT_DIR}/final_model.pt", "--port", "7860"],
                              stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    app_rows = []
    try:
        from gradio_client import Client, handle_file
        client = None
        for _ in range(90):
            try: client = Client("http://127.0.0.1:7860/", verbose=False); break
            except Exception: time.sleep(2)
        assert client is not None, "app did not start"
        for i in sel:
            t1 = time.time(); r = client.predict(handle_file(f"app_test/{i}.png"), api_name="/predict")
            app_rows.append(dict(image=int(i), true=CLASSES[yte_np[i]], app_top1=r["label"], notebook_top1=CLASSES[final_pred[i]],
                                 app_top3=[c["label"] for c in r["confidences"]], latency_s=time.time() - t1))
    finally:
        server.terminate()
    app_df = pd.DataFrame(app_rows); app_df.to_csv(f"{OUT_DIR}/final_webapp_test.csv", index=False)
    print(f"Web-app test on {len(app_df)} uploaded CIFAR-10 test images (final model: {FINAL_METHOD}):")
    print(f"  app top-1 matches notebook prediction: {(app_df.app_top1 == app_df.notebook_top1).mean()*100:.0f}%")
    print(f"  app top-1 accuracy: {(app_df.app_top1 == app_df.true).mean()*100:.0f}% | top-3 accuracy: "
          f"{np.mean([t in a for t, a in zip(app_df.true, app_df.app_top3)])*100:.0f}%")
    print(f"  median response time: {app_df.latency_s.median()*1000:.0f} ms per image")

## 11. Record the exact software and hardware used
Writes `outputs/environment.json` and overwrites `requirements.txt` with the exact versions from this run.

In [ ]:
from importlib.metadata import version, PackageNotFoundError
PKGS = ["torch", "torchvision", "numpy", "pandas", "matplotlib", "scikit-learn", "scipy", "pillow", "gradio", "gradio_client"]
vers = {}
for p in PKGS:
    try: vers[p] = version(p)
    except PackageNotFoundError: pass
env = dict(python=platform.python_version(), packages=vers, device=str(device),
           gpu=torch.cuda.get_device_name(0) if device.type == "cuda" else None, cuda=torch.version.cuda,
           data_source=DATA_SOURCE, data_id=DATA_ID, quick_test=QUICK_TEST, total_runs=len(df))
json.dump(env, open(f"{OUT_DIR}/environment.json", "w"), indent=2)
if not QUICK_TEST:
    with open("requirements.txt", "w") as f:
        f.write("# Exact versions recorded by the final notebook run\n")
        for p, v in vers.items(): f.write(f"{p.split('+')[0]}=={v.split('+')[0]}\n")
print(json.dumps(env, indent=2)); print(sorted(os.listdir(OUT_DIR)))

## External data, code, models and AI assistance
* **Data:** CIFAR-10 (Krizhevsky, 2009), https://www.cs.toronto.edu/~kriz/cifar.html
* **Pretrained model:** ResNet-18 ImageNet-1k weights `resnet18_a1_0-d63eafa0.pth` from timm (Wightman, Touvron & Jégou, 2021), https://github.com/huggingface/pytorch-image-models (Apache-2.0)
* **Libraries:** PyTorch, torchvision, NumPy, pandas, Matplotlib, scikit-learn, SciPy, Pillow, Gradio, gradio_client
* **AI-assisted tool:** Claude (Anthropic) was used to help write the notebook code, the web app, the phone-photo test script and the README.